<a href="https://colab.research.google.com/github/josep-prog/AmaliTech-DEG-Backend_Project/blob/main/Kinyarwanda_TTS_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kinyarwanda TTS fine-tuning (MMS + finetune-hf-vits)

Fine-tunes `facebook/mms-tts-kin` on our own transcribed podcast/documentary audio, using
[ylacombe/finetune-hf-vits](https://github.com/ylacombe/finetune-hf-vits).

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (free tier). This model is small
(83M params) -- the repo's own docs say a full fine-tune can take as little as ~20 minutes with
80-150 samples, so a free-tier session is normally plenty. We still checkpoint to Google Drive
as a safety net against disconnects.

**Data**: this notebook expects a dataset already pushed to the HF Hub by `build_hf_dataset.py`
(run locally, from the machine that has the raw audio/transcripts -- see that script for how to
push it). Set `HF_DATASET_REPO` below to whatever repo id you pushed it to.

In [1]:
!nvidia-smi

Thu Sep 17 03:41:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Mount Drive (checkpoint persistence across disconnects)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/kinyarwanda_tts_finetune'
os.makedirs(DRIVE_DIR, exist_ok=True)

Mounted at /content/drive


In [3]:
!python3 -c "import os; print(os.listdir('/content/drive/MyDrive')[:20])"

['APPLICATION FOR EQUATING FOREIGN QUALIFICATIONS - GENERAL EDUCATION - B240119134639YISG.pdf', 'The Weight of Time: A Journey Through Fear and Self-Discovery.gdoc', 'first part.gdoc', 'second part of presentation.gdoc', 'Untitled document (18).gdoc', 'Untitled document (17).gdoc', 'Joseph (a proof of bank account).pdf', 'GITHUB TOKEN.gdoc', 'Untitled document (16).gdoc', 'Untitled document (15).gdoc', 'Untitled document (14).gdoc', 'book-pages', 'CRIDENTIALS.gdoc', 'articles.gdoc', 'Project Checkpoint 1 - Submittable Worksheet.gdoc', 'Project Checkpoint 2 - Submittable Worksheet.gdoc', 'milestone.gdoc', 'bV Lean Canvas Workbook 25-26 - template.gslides', 'Module1 Journey.gdoc', 'Untitled document (13).gdoc']


## 2. Install finetune-hf-vits and its requirements

In [4]:
!git clone https://github.com/ylacombe/finetune-hf-vits.git
%cd finetune-hf-vits
!pip install -q -r requirements.txt
!pip install -q accelerate -U

# Cython monotonic alignment search -- required, the pure-Python fallback is far too slow.
%cd monotonic_align
!mkdir -p monotonic_align
!python setup.py build_ext --inplace
%cd ..

Cloning into 'finetune-hf-vits'...
remote: Enumerating objects: 173, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 173 (delta 81), reused 74 (delta 74), pack-reused 78 (from 1)
Receiving objects: 100% (173/173), 1.24 MiB | 34.22 MiB/s, done.
Resolving deltas: 100% (99/99), done.
/content/finetune-hf-vits
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 1.7 MB/s eta 0:00:00
/content/finetune-hf-vits/monotonic_align
Compiling core.pyx because it changed.
[1/1] Cythonizing core.pyx
/usr/local/lib/python3.13/dist-packages/Cython/Compiler/Main.py:381: FutureWarning: Cython directive 'language_level' not set, using '3str' for now (Py3). This has changed from earlier releases! File: /content/finetune-hf-vits/monotonic_align/core.pyx
  tree = Parsing.p_module(s, pxd, full_module_name)
performance hint: core.pyx:7:5: Exception check on 'maximum_path_each' will always require the GIL to be acquired.
Possible solutions

## 3. Log in to Hugging Face
Needs a **write**-scoped token from https://huggingface.co/settings/tokens -- used to pull the
dataset (if private) and push the converted checkpoint + fine-tuned model.

In [5]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


## 4. Convert the MMS checkpoint (one-time, adds a trainable discriminator)

`facebook/mms-tts-kin` ships inference-only. Fine-tuning needs the GAN discriminator too, so we
convert it once and push the result to our own namespace. Re-running this is harmless (it just
re-creates the same starting point) but unnecessary once you have `HF_USERNAME/mms-tts-kin-train`.

In [9]:
HF_USERNAME = 'Joshnish'
CONVERTED_CHECKPOINT_REPO = f'{HF_USERNAME}/mms-tts-kin-train'

!python convert_original_discriminator_checkpoint.py \
    --language_code kin \
    --pytorch_dump_folder_path ./mms-tts-kin-train \
    --push_to_hub {CONVERTED_CHECKPOINT_REPO}

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
2026-09-17 04:03:13.559563: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--facebook--mms-tts-kin/snapshots/71d00a03419b9bfd203a6bfa4d6bc565cbe3c4e0/config.json
Model config VitsConfig {
  "activation_dropout": 0.1,
  "architectures": [
    "VitsModel"
  ],
  "attention_dropout": 0.1,
  "depth_separable_channels": 2,
  "depth_separable_num_layers": 3,
  "discriminator_kernel_size": 5,
  "discriminator_period_cha

In [8]:
!pip install -q "transformers==4.46.3"
import transformers
print(transformers.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 80.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
5.16.1


## 5. Write the training config

Base hyperparameters follow the repo's own `finetune_mms.json` example, sized down for a T4's
~15GB VRAM. `mwanafunzi` and `mwanafunzi2` are confirmed to be the same narrator, so both are
trained on together as one voice (no speaker filtering needed) -- this is just the ~1,300 clips
transcribed so far as a first smoke-test run of the pipeline; rerun `build_hf_dataset.py` and
this notebook later once more of the ~80 hours has been transcribed.

In [19]:
import json

HF_DATASET_REPO = 'Joshnish/kinyarwanda-tts-data'
FINETUNED_REPO = f'{HF_USERNAME}/mms-tts-kin-finetuned'

config = {
    'project_name': 'kinyarwanda_mms_finetuning',
    'push_to_hub': True,
    'hub_model_id': FINETUNED_REPO,
    'report_to': [],
    'overwrite_output_dir': False,
    'output_dir': f'{DRIVE_DIR}/checkpoints',

    'dataset_name': HF_DATASET_REPO,
    'audio_column_name': 'audio',
    'text_column_name': 'text',
    'train_split_name': 'train',
    'eval_split_name': 'test',
    'speaker_id_column_name': 'speaker_id',

    'full_generation_sample_text': "Muraho, amakuru yanyu?",

    'max_duration_in_seconds': 20,
    'min_duration_in_seconds': 1.0,
    'max_tokens_length': 500,

    'model_name_or_path': f'./mms-tts-kin-train',

    'preprocessing_num_workers': 2,

    'do_train': True,
    'num_train_epochs': 100,
    'gradient_accumulation_steps': 2,
    'gradient_checkpointing': True,
    'per_device_train_batch_size': 8,
    'learning_rate': 2e-5,
    'adam_beta1': 0.8,
    'adam_beta2': 0.99,
    'warmup_ratio': 0.01,
    'group_by_length': True,

    'do_eval': True,
    'eval_steps': 500,
    'per_device_eval_batch_size': 8,
    'max_eval_samples': 25,
    'do_step_schedule_per_epoch': True,

    'save_steps': 500,
    'save_total_limit': 3,

    'weight_disc': 3,
    'weight_fmaps': 1,
    'weight_gen': 1,
    'weight_kl': 1.5,
    'weight_duration': 1,
    'weight_mel': 35,

    'fp16': True,
    'seed': 456,
}

with open('training_config_kinyarwanda.json', 'w') as f:
    json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))

{
  "project_name": "kinyarwanda_mms_finetuning",
  "push_to_hub": true,
  "hub_model_id": "Joshnish/mms-tts-kin-finetuned",
  "report_to": [],
  "overwrite_output_dir": false,
  "output_dir": "/content/drive/MyDrive/kinyarwanda_tts_finetune/checkpoints",
  "dataset_name": "Joshnish/kinyarwanda-tts-data",
  "audio_column_name": "audio",
  "text_column_name": "text",
  "train_split_name": "train",
  "eval_split_name": "test",
  "speaker_id_column_name": "speaker_id",
  "full_generation_sample_text": "Muraho, amakuru yanyu?",
  "max_duration_in_seconds": 20,
  "min_duration_in_seconds": 1.0,
  "max_tokens_length": 500,
  "model_name_or_path": "./mms-tts-kin-train",
  "preprocessing_num_workers": 2,
  "do_train": true,
  "num_train_epochs": 100,
  "gradient_accumulation_steps": 2,
  "gradient_checkpointing": true,
  "per_device_train_batch_size": 8,
  "learning_rate": 2e-05,
  "adam_beta1": 0.8,
  "adam_beta2": 0.99,
  "warmup_ratio": 0.01,
  "group_by_length": true,
  "do_eval": true,
  

## 6. Resume support
If `output_dir` (on Drive) already has checkpoints from a previous, disconnected session, this
picks up from the latest one automatically instead of restarting from scratch.

In [20]:
import json, os

ckpt_dir = config['output_dir']
has_checkpoints = os.path.isdir(ckpt_dir) and any(d.startswith('checkpoint') for d in os.listdir(ckpt_dir))

with open('training_config_kinyarwanda.json') as f:
    cfg = json.load(f)
cfg['resume_from_checkpoint'] = 'latest' if has_checkpoints else None
cfg['overwrite_output_dir'] = not has_checkpoints
with open('training_config_kinyarwanda.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print('Resuming from latest checkpoint' if has_checkpoints else 'Starting fresh run')

Starting fresh run


## 7. Train

In [17]:
%%writefile fix_dataset.py
import os
from datasets import load_dataset

HF_DATASET_REPO = os.environ["HF_DATASET_REPO"]
ds = load_dataset(HF_DATASET_REPO)
ds = ds.map(lambda x: {"speaker_id": 0})
ds.push_to_hub(HF_DATASET_REPO)
print("done")

Writing fix_dataset.py


In [22]:
!grep -n "speaker_id" /content/finetune-hf-vits/run_vits_finetuning.py

110:                "If `True` and if `speaker_id_column_name` is specified, it will replace current speaker embeddings with a new set of speaker embeddings."
204:    speaker_id_column_name: str = field(
218:    filter_on_speaker_id: int = field(
222:                "If `speaker_id_column_name` and `filter_on_speaker_id` are set, will filter the dataset to keep a single speaker_id (`filter_on_speaker_id`)  "
380:        batch["speaker_id"] = (
381:            torch.tensor([feature["speaker_id"] for feature in features]) if "speaker_id" in features[0] else None
623:        data_args.speaker_id_column_name is not None
624:        and data_args.speaker_id_column_name not in next(iter(raw_datasets.values())).column_names
627:            f"--speaker_id_column_name {data_args.speaker_id_column_name} not found in dataset '{data_args.speaker_id_column_name}'. "
628:            "Make sure to set `--speaker_id_column_name` to the correct text column - one of "
679:    speaker_id_column_name = da

In [23]:
!python -c "from datasets import load_dataset; d = load_dataset('Joshnish/kinyarwanda-tts-data', download_mode='force_redownload'); print(d); print(d['train'][0].keys()); print(d['train'][0]['speaker_id'])"

data/train-00000-of-00001.parquet: 100% 331M/331M [00:02<00:00, 111MB/s]
data/test-00000-of-00001.parquet: 100% 7.20M/7.20M [00:00<00:00, 14.3MB/s]
Generating train split: 100% 1286/1286 [00:01<00:00, 710.71 examples/s]
Generating test split: 100% 27/27 [00:00<00:00, 997.83 examples/s]
DatasetDict({
    train: Dataset({
        features: ['audio', 'text', 'speaker_id'],
        num_rows: 1286
    })
    test: Dataset({
        features: ['audio', 'text', 'speaker_id'],
        num_rows: 27
    })
})
dict_keys(['audio', 'text', 'speaker_id'])
0


In [18]:
%env HF_DATASET_REPO=Joshnish/kinyarwanda-tts-data
!python fix_dataset.py

env: HF_DATASET_REPO=Joshnish/kinyarwanda-tts-data
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Uploading the dataset shards:   0% 0/1 [00:00<?, ? shards/s]
Map:   0% 0/1286 [00:00<?, ? examples/s]
Map:  30% 388/1286 [00:00<00:00, 1671.50 examples/s]
Map:  60% 776/1286 [00:00<00:00, 1283.53 examples/s]
Map: 100% 1286/1286 [00:01<00:00, 1096.92 examples/s]

Creating parquet from Arrow format:   0% 0/4 [00:00<?, ?ba/s]
Creating parquet from Arrow format:  25% 1/4 [00:00<00:00,  6.00ba/s]
Creating parquet from Arrow format:  50% 2/4 [00:00<00:00,  2.35ba/s]
Creating parquet from Arrow format: 100% 4/4 [00:06<00:00,  1.63s/ba]
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpb5wcfvwf.parquet    :   5% 16.0M/331M [00:00<?, ?B/s]

Processing Files (0 / 1)      :   5% 16.0M/331M [00:02<00:41, 7.55MB/s, 1.51MB/s  ]

Proce

In [14]:
import datasets, huggingface_hub
print("datasets", datasets.__version__)
print("huggingface_hub", huggingface_hub.__version__)

datasets 4.8.5
huggingface_hub 1.29.0


In [15]:
!pip install -q -U datasets
import importlib, datasets
importlib.reload(datasets)
print(datasets.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 35.9 MB/s eta 0:00:00
5.0.1


In [31]:
!accelerate launch run_vits_finetuning.py training_config_kinyarwanda.json

Streaming output truncated to the last 5000 lines.
Exception ignored in: <_io.BytesIO object at 0x79c71818d7b0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/pickling_support.py", line 13, in unpickle_traceback
    def unpickle_traceback(tb_frame, tb_lineno, tb_next):
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x79c71818c2c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/pickling_support.py", line 13, in unpickle_traceback
    def unpickle_traceback(tb_frame, tb_lineno, tb_next):
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x79c71818d9e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/pickling_support.py", line 13, in unpickle_traceback
    def unpickle_traceback(tb_frame, tb_lineno, tb_next):
BufferError: Existing exports of dat

## 8. Try it out
Loads the fine-tuned model straight from `output_dir` (no need to wait for the final Hub push)
and synthesizes a few sentences -- copy in real trouble-spot lines from `sentences.txt` in the
main project repo to judge intonation/naturalness the same way `compare.py` does locally.

In [ ]:
import torch
from transformers import VitsModel, VitsTokenizer
import IPython.display as ipd

model_path = config['output_dir']  # or FINETUNED_REPO once push_to_hub has run at the end
model = VitsModel.from_pretrained(model_path)
tokenizer = VitsTokenizer.from_pretrained(model_path)

test_sentences = [
    "Muraho, amakuru yanyu?",
    "Nk'uko byatangajwe, imvura izagwa mu turere twinshi tw'igihugu.",
    "Urakoze cyane kubw'ubufasha wampaye, nzabimenyesha abandi.",
]

for text in test_sentences:
    inputs = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        waveform = model(**inputs).waveform
    print(text)
    display(ipd.Audio(waveform.numpy(), rate=model.config.sampling_rate))

In [24]:
!python -c "from datasets import load_dataset; d = load_dataset('Joshnish/kinyarwanda-tts-data', download_mode='force_redownload'); print(d); print(sorted(d['train'][0].keys())); print(d['train'][0]['speaker_id'])"

data/train-00000-of-00001.parquet: 100% 331M/331M [00:04<00:00, 77.6MB/s]
data/test-00000-of-00001.parquet: 100% 7.20M/7.20M [00:00<00:00, 15.7MB/s]
Generating train split: 100% 1286/1286 [00:01<00:00, 1038.55 examples/s]
Generating test split: 100% 27/27 [00:00<00:00, 706.72 examples/s]
DatasetDict({
    train: Dataset({
        features: ['audio', 'text', 'speaker_id'],
        num_rows: 1286
    })
    test: Dataset({
        features: ['audio', 'text', 'speaker_id'],
        num_rows: 27
    })
})
['audio', 'speaker_id', 'text']
0


In [25]:
!sed -n '695,830p' /content/finetune-hf-vits/run_vits_finetuning.py

    # return attention_mask for Vits models
    forward_attention_mask = True

    with training_args.main_process_first(desc="select range of samples"):
        if data_args.max_train_samples is not None:
            raw_datasets["train"] = raw_datasets["train"].select(range(data_args.max_train_samples))

        if data_args.max_eval_samples is not None:
            raw_datasets["eval"] = raw_datasets["eval"].select(range(data_args.max_eval_samples))

    speaker_id_dict = {}
    new_num_speakers = 0
    if speaker_id_column_name is not None:
        if training_args.do_train:
            # if filter_on_speaker_id, filter so that we keep only the speaker id
            if filter_on_speaker_id is not None:
                with training_args.main_process_first(desc="filter speaker id"):
                    raw_datasets["train"] = raw_datasets["train"].filter(
                        lambda speaker_id: (speaker_id == filter_on_speaker_id),
                        num_proc=num_workers,
 

In [27]:
import re
lines = open('/content/finetune-hf-vits/run_vits_finetuning.py').readlines()
pat = re.compile(r'DataCollatorTTSWithPadding|forward_attention_mask|new_num_speakers =|DataLoader')
for i, l in enumerate(lines, 1):
    if pat.search(l):
            print(i, l.rstrip())

300 class DataCollatorTTSWithPadding:
308         forward_attention_mask (`bool`)
314     forward_attention_mask: bool
355         batch = self.tokenizer.pad(input_ids, return_tensors="pt", return_attention_mask=self.forward_attention_mask)
696     forward_attention_mask = True
706     new_num_speakers = 0
722                 new_num_speakers = len(speaker_id_dict)
828             elif new_num_speakers == 1:
851     data_collator = DataCollatorTTSWithPadding(
854         forward_attention_mask=forward_attention_mask,
905         train_dataloader = torch.utils.data.DataLoader(
926         eval_dataloader = torch.utils.data.DataLoader(


In [29]:
print("".join(lines[729:770]))

            return_attention_mask=False,
            do_normalize=do_normalize,
        )

        batch["labels"] = audio_inputs.get("input_features")[0]

        # process text inputs
        input_str = batch[text_column_name].lower() if do_lower_case else batch[text_column_name]
        
        if is_uroman:
            input_str = uromanize(input_str, uroman_path=uroman_path)
        string_inputs = tokenizer(input_str, return_attention_mask=False)

        batch[model_input_name] = string_inputs.get("input_ids")[: max_tokens_length + 1]
        batch["waveform_input_length"] = len(sample["array"])
        batch["tokens_input_length"] = len(batch[model_input_name])
        batch["waveform"] = batch[audio_column_name]["array"]

        batch["mel_scaled_input_features"] = audio_inputs.get("mel_scaled_input_features")[0]

        if speaker_id_column_name is not None:
            if new_num_speakers > 1:
                # align speaker_id to [0, num_speaker_id-1].
                b

In [30]:
!sed -i '1099s/batch\["speaker_id"\]/batch.get("speaker_id")/' /content/finetune-hf-vits/run_vits_finetuning.py
!sed -i '1288s/batch\["speaker_id"\]/batch.get("speaker_id")/' /content/finetune-hf-vits/run_vits_finetuning.py
!sed -i '1390s/batch\["speaker_id"\]/batch.get("speaker_id")/' /content/finetune-hf-vits/run_vits_finetuning.py
!sed -n '1099p;1288p;1390p' /content/finetune-hf-vits/run_vits_finetuning.py

                    speaker_id=batch.get("speaker_id"),
                            speaker_id=batch.get("speaker_id"),
                        speaker_id=batch.get("speaker_id"),
